# 5B Build a Governed AI Playground

**Purpose:** Build a useful AI application in the remote VS Code workspace from [Part 05](05_Running_VS_Code.ipynb).

**What we're doing:** Run a small Gradio interface, swap gateway aliases, inspect latency and token usage, and optionally launch the same script through Klique's Gradio Launcher.

**Why:** The application stays the same when the permitted model changes. Governance comes from the LLM Gateway, not from the user interface.

**Output:** A working single-turn playground, a request visible in AI Usage, and a saved configuration without committed secrets.

**Time:** About 30–45 minutes, plus environment setup. CPU is sufficient; the model runs behind your gateway.

> **Before you start:** Finish Part 05. For live calls, complete Section 02 with a working gateway token, profile and one or more model aliases. Use an administrator-approved Python 3.10+ image and package source. Browser access requires an administrator-confirmed preview route or the optional Gradio Launcher. A remote loopback address is not your laptop's loopback address.

This is a **guided Markdown lab**. Run shell commands in the **remote VS Code terminal**, from the repository root. It deliberately does not start a public server when you open the notebook.

## 5B.1 Prepare a Reproducible Workspace

Use the existing checkout under your configured persistent workspace. Do not clone a second copy. In its root:

```bash
pwd
git status --short
python --version
python -m venv .venv
source .venv/bin/activate
python -m pip install -r 05_VSCode/requirements.txt
python -m pip check
```

If `.venv` already exists, activate and inspect it rather than overwriting it. These are POSIX remote-terminal commands; your image must provide venv support. Select this environment with **Python: Select Interpreter** in VS Code. Use an approved image or ask the administrator if packages cannot be installed.

Open [`playground.py`](playground.py) and [`gateway_client.py`](gateway_client.py). The first owns the UI; the second performs exactly one HTTPS request per call. Unlike root `config.py`, the lab helper supports environment-only deployments, preserves injected-variable precedence, and resolves `.env` relative to the repository rather than the current terminal directory.

> **Checkpoint:** Dependency checks pass. No model request has been made.

## 5B.2 Prove the Wiring Without Credentials

```bash
python 05_VSCode/playground.py --check
python -m unittest discover -s 05_VSCode -p 'test_gateway_playground.py' -v
```

Expected: an answer beginning **OFFLINE FIXTURE**, `offline_fixture: true`, and `usage_reported: null`. The fixture is authored text, not a generated answer. Tests check the exact gateway route, headers, alias selection, errors, no retries, secret redaction, and UI construction.

The callback, `answer()`, returns answer text plus metadata. Put a breakpoint on its call to `client.complete()` and run the fixture with VS Code's Python debugger. Inspect `prompt` and `alias`. Do not display a live token or expand the client's configuration in a screenshot.

> **Checkpoint:** The callback works without a gateway, and no fake token usage is presented as measured usage.

## 5B.3 Configure the Governed Endpoint

Edit the ignored **repository-root `.env`** privately. Copy field names from [`gateway.env.example`](gateway.env.example), not real values into the tracked example. Keep the Section 01 values if they are already there, then add:

```text
KLIQUE_PROFILE=your-profile-name
KLIQUE_MODEL_ALIASES=your-first-profile-alias,your-second-profile-alias
```

The shared helper also needs `KLIQUE_BASE_URL` and `KLIQUE_GATEWAY_TOKEN` from Section 01. It derives `https://<same-host>/llm`, then calls `/v1/chat/completions` with **Authorization: Bearer** and **X-ClearML-Profile**. Alternatively inject `KLIQUE_LLM_GATEWAY_URL=https://<gateway-host>/llm` explicitly.

The aliases must be configured in that profile. They are **not necessarily** `KLIQUE_MODEL`, which names the direct endpoint. The gateway does not support `/v1/models` discovery in the documented lab flow, so this UI uses your explicit allowlist.

Do not paste tokens into source, notebook cells, shell arguments, screenshots or Git. An ignored file can still be included in a workspace snapshot: use approved secret injection for deployment and review snapshot access/retention with your administrator. `.gitignore` is not encryption.

When you are ready for **one real request** using a public sample prompt:

```bash
python 05_VSCode/playground.py --live --check
```

Expected: answer, requested alias, returned model ID if supplied, client elapsed seconds, reported usage, and finish reason. `null` means absent, not zero. Some models need more output tokens or do not support every sampling option; the lab does not automatically retry or send provider-specific arguments.

> **Checkpoint:** A single live request succeeds. On 429, stop and inspect your rate limit and budget. Both can produce 429. Optional fingerprint codes belong in `.env` only after observing them in Section 03.

## 5B.4 Start the UI Without Publishing It

In **remote terminal A**:

```bash
python 05_VSCode/playground.py --live
```

In **remote terminal B**, with the same session:

```bash
curl --fail http://127.0.0.1:7860/
```

The script binds **127.0.0.1** and uses **share=False**. The curl check proves the server runs inside the remote environment. It does not prove your laptop can reach it.

Use only the preview/access route your administrator has confirmed for this VS Code deployment. The Klique VS Code documentation does not promise arbitrary port forwarding, so this lab does not assume a Ports tab, public link, or generic SSH tunnel. If no supported route exists, keep the terminal check and use the optional Gradio Launcher below. Do not enable a public Gradio share link or change network/security settings as a workaround.

In the browser, send one question, change the alias, and send it again. Try temperatures 0 and 0.7 without changing both alias and temperature in the same comparison. The UI is deliberately single-turn: it does not keep prior chats or transmit an unbounded conversation.

> **Screenshot placeholder:** The playground with alias selector, public sample answer and request metadata. No token or private prompt visible.

<!-- screenshot: images/vscode_playground.png (Governed playground with safe sample prompt and metadata.) -->

## 5B.5 Read the Evidence

Each **Send one request** click makes at most one request. The queue limits pending work to eight and concurrency to one, but that is not a spending cap. Repeated clicks still cause repeated calls; stop after the planned two or three comparisons.

| Field | Meaning |
|---|---|
| requested_alias | Profile alias selected by the user |
| returned_model | Backend-reported identifier; it may differ from the alias |
| client_elapsed_seconds | End-to-end client wall time, including transport and queueing, not model throughput |
| usage_reported | Provider/gateway-reported usage, or null when missing |
| finish_reason | For example stop or length; inspect truncation before judging quality |

Open the [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage), choose **Gateway**, and select the time window of your run. Compare request count and reported tokens. Do not invent dollar costs when Resource pricing is absent.

App-access authentication and model-call authentication are separate. Even when the Gradio Launcher authenticates each viewer, this sample uses **one server-side gateway token**. Usage is attributed to that identity/project, not automatically to the viewer. Keep the optional deployment private for this exercise.

> **Checkpoint:** You can explain which model was requested, what timing measures, and whose identity pays for each request.

## 5B.6 Optional: Run the Script with Gradio Launcher

This is a separate managed application, not automatic deployment from VS Code. See the [Gradio Launcher reference](https://klique.ai/docs/latest/docs/webapp/applications/apps_gradio).

1. Ensure the script and its dependencies are available in a branch you are permitted to use. A local commit is not visible to the launcher until pushed to an authorized repository. Do not include `.env` or private result files.
2. Ask your administrator to provide an approved image with `requirements.txt` installed and securely inject the four gateway settings. The launch form's documented fields do **not** establish a secret-entry mechanism, so do not put a token into **Arguments**.
3. Open **Applications → UI Dev → Gradio Launcher → Launch New**. Set **Git Repository** to this repository (or your authorized fork), **Git Branch** to the branch containing these files, and **Gradio Script Name** to `05_VSCode/playground.py`.
4. Choose your approved **Queue**, **Docker Image**, project and task name. Set **Arguments** to `--live`, plus only the host/port arguments your administrator says the authenticated App Gateway requires. The script supports `--host 0.0.0.0`, but use it only in that controlled launcher network, never as a generic preview workaround.
5. Set **Idle Time Limit** to 1 hour for the lab. Leave **Share Access** empty so only you and tenant administrators can see the instance.
6. Launch, inspect **Console Log**, and open the documented **Gradio App** link. This link depends on the Klique AI Application Gateway. Test one public prompt and verify AI Usage.

Do not assume the launcher installs the requirements file, copies VS Code's `.env`, chooses the right bind address, or inherits your IDE's environment. Verify each with your administrator. Avoid sharing access until you have designed per-user attribution and an appropriate budget.

> **Checkpoint:** The link works through the authenticated application route, and no gateway secret appears in the form or logs.

## 5B.7 Save and Stop

1. Save source changes and inspect `git status --short`. Keep tokens and outputs out of commits.
2. In remote terminal A, press **Ctrl+C**. Repeat the remote curl check; a fresh request should fail after the server stops.
3. If you launched a Gradio instance, stop that instance separately and verify stopped status.
4. Record the VS Code session ID. Stop the IDE when finished, or leave it running only while proceeding directly to the next lab.
5. After a later restore, select/recreate the environment and restart the script explicitly. Saved files survive according to your workspace policy; the process does not.

The model endpoint has its own lifecycle. Stop it only if you created it for this exercise and no one else depends on it.

## Troubleshooting

| Symptom | Check |
|---|---|
| 401/403 | Correct gateway token and profile access; do not print the token |
| 404 | Alias exists in the selected profile; use `/llm/v1/chat/completions` |
| 429 | Rate limit or wallet budget; inspect Section 03 rather than guessing or retrying |
| Remote curl works but browser fails | Approved preview/App Gateway route, not a public share workaround |
| Port already in use | Identify your own existing server; stop it or choose a different approved port |
| Launcher starts but cannot configure client | Approved secret injection, correct branch and dependencies |
| Response has no usage or model ID | Preserve null and compare gateway-side usage; do not fabricate values |
| Timeout | Request may have executed; inspect AI Usage before resending |

**Next:** [Chat with This Repository](05c_Chat_with_This_Repository.ipynb)  
**Back:** [VS Code introduction](05_Running_VS_Code.ipynb)  
**Index:** [README](../README.md)